[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/01_wgcna.ipynb )

# R1-Q2 Week 2 — Build per-tissue co-expression networks

This notebook builds two co-expression networks — one for shoot, one for root — using PyWGCNA and the parameters you pre-committed in Week 1.

By the end of this notebook you will have:

- Two per-tissue WGCNA networks built with your pre-committed parameters (`shoot_wgcna.pkl`, `root_wgcna.pkl`).
- Soft-thresholding power diagnostics and module-structure inspection plots for each tissue.
- A `network_summary.json` (per-tissue gene count, module count, soft power used, key parameters) ready as Week 2 EQ#2 input.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

#!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory,
# and load the pre-commit you wrote in Notebook 0.
import json
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

precommit_path = OUTPUT_DIR / "precommit.json"
try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find your pre-commit file.\n"
        f"  Expected at: {precommit_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes `precommit.json` to this location.\n"
    ) from None

print(f"Loaded pre-commit with {len(precommit)} top-level entries.")

## 1. Load the filtered matrices and confirm your pre-committed parameters

In `00_question_orientation.ipynb` you produced two filtered expression matrices — one for shoot samples, one for root — and wrote your network-construction choices to `precommit.json`. This section loads both back into memory and prints your committed choices so they're visible alongside the code that will use them.

This is a deliberate beat, not bookkeeping. The whole point of pre-committing parameters in writing is that you can see them when the analysis runs. If something in the printed summary surprises you, stop and re-read what you wrote in N0 before continuing.

In [ ]:
# Load the filtered shoot and root matrices written by Notebook 0.
import pandas as pd

shoot_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_path  = OUTPUT_DIR / "root_filtered.parquet"

try:
    shoot = pd.read_parquet(shoot_path)
    root  = pd.read_parquet(root_path)
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"\nCould not find a filtered matrix.\n"
        f"  Expected: {shoot_path}\n"
        f"            {root_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes both files to this location.\n"
    ) from None

In [ ]:
# Confirm matrix shapes. Each matrix should be samples (rows) by genes (columns).
print("Shoot matrix:")
print(f"  {shoot.shape[0]} samples  ×  {shoot.shape[1]} genes")
print()
print("Root matrix:")
print(f"  {root.shape[0]} samples  ×  {root.shape[1]} genes")

The two matrices should have **the same number of genes** (filtering in N0 was applied to both with the same criteria) but **different sample counts** (shoot and root samples are not paired). If either of those expectations is broken, go back to N0 and inspect the filter step before going further — the rest of this notebook assumes both matrices are well-formed.

In [ ]:
# Echo the pre-committed network-construction choices you wrote in N0.
nc = precommit["network_construction"]
p  = nc["params"]

print("Pre-committed network-construction choices")
print("-" * 44)
print(f"Method:                {nc['method']}")
print(f"Network type:          {p['networkType']}")
print(f"TOM type:              {p['TOMType']}")
print(f"R² cutoff (scale-free): {p['RsquaredCut']}")
print(f"Minimum module size:   {p['minModuleSize']}")
print(f"Module merge threshold: {p['MEDissThres']}")
print()
print("Reasoning:")
print(nc["reasoning"])

These are the choices you'll feed PyWGCNA in section 2. The soft-thresholding power isn't fixed yet — PyWGCNA will scan a default range of candidate powers and pick the lowest one that achieves scale-free topology at the R² cutoff you pre-committed (0.9). Section 2 will record the chosen power in both `precommit.json` (under `network_construction.applied`) and `network_summary.json` (for Week 2 EQ#2).

With matrices loaded and parameters confirmed, you're ready to build the networks.

## 2. Build the per-tissue co-expression networks

Now you'll build two co-expression networks using PyWGCNA — one for shoot samples, one for root samples — feeding each the parameters you pre-committed.

PyWGCNA's `findModules()` does a lot of work in a single call: it scans candidate soft-thresholding powers, picks the one that achieves your R² cutoff, builds the adjacency matrix and topological overlap matrix, detects modules via dynamic tree cut, and merges modules whose eigengenes are similar enough. Expect each tissue to take a few minutes on Colab; you'll see progress messages as each step runs, plus a diagnostic plot showing the soft-power selection.

The shoot and root networks are built independently. There's no shared state between them.

In [ ]:
# Shape check before calling PyWGCNA.
# PyWGCNA expects samples in rows, genes in columns. If the matrices were
# accidentally transposed in N0, PyWGCNA will produce confusing errors deep
# in the call chain. Better to fail loudly here.
assert shoot.shape[0] < shoot.shape[1], (
    f"Shoot matrix looks transposed: {shoot.shape}. "
    f"Expected samples (rows) < genes (columns)."
)
assert root.shape[0] < root.shape[1], (
    f"Root matrix looks transposed: {root.shape}. "
    f"Expected samples (rows) < genes (columns)."
)
print("Shape orientation check passed.")

In [ ]:
# Build the shoot network.
import PyWGCNA

shoot_wgcna = PyWGCNA.WGCNA(
    name="shoot",
    species="arabidopsis thaliana",
    geneExp=shoot,
    outputPath=str(OUTPUT_DIR) + "/",
    networkType=p["networkType"],
    TOMType=p["TOMType"],
    RsquaredCut=p["RsquaredCut"],
    minModuleSize=p["minModuleSize"],
    MEDissThres=p["MEDissThres"],
    save=False,  # we save deliberately in S4, not as a side effect here
)

shoot_wgcna.findModules()

In [ ]:
# Build the root network.
root_wgcna = PyWGCNA.WGCNA(
    name="root",
    species="arabidopsis thaliana",
    geneExp=root,
    outputPath=str(OUTPUT_DIR) + "/",
    networkType=p["networkType"],
    TOMType=p["TOMType"],
    RsquaredCut=p["RsquaredCut"],
    minModuleSize=p["minModuleSize"],
    MEDissThres=p["MEDissThres"],
    save=False,
)

root_wgcna.findModules()

Both networks are built. Each `WGCNA` object now carries the soft-thresholding power that was selected, the adjacency matrix, the topological overlap matrix, the module assignments per gene, and the module eigengenes. The next section records the soft-power that PyWGCNA chose into your pre-commit file and into a fresh `network_summary.json` for Week 2 EQ#2.